# Data inventory — record counts across every source and layer

Lists **every data source we have, in every medallion layer**, with the number of
records in each.

- **Bronze** is raw Parquet on disk under `data/bronze/<source>/` (one group per source).
- **Silver** and **Gold** live in the **DuckLake catalog** (dbt staging views,
  intermediate + canonical tables, gold marts), plus the Python-written Parquet
  under `data/silver/` and the gold Parquet exports under `data/gold/`.

This is the one notebook that reads the DuckLake catalog directly: the silver
canonical tables (`team`, `match`, the link tables, …) aren't all exported to
Parquet, so a Parquet-only inventory couldn't see them. The reads are `count(*)`
only — DuckLake is concurrency-safe for readers, so this never blocks a dbt run.

> Note: layers double-count by design (a silver staging view re-derives its bronze
> source; a gold export duplicates its catalog table), so there is no meaningful
> grand total — read the per-source counts and the per-layer subtotals.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/app/data"))
CATALOG = os.environ.get(
    "POSTGRES_CATALOG_URL",
    "postgres:dbname=ducklake user=ducklake password=ducklake host=ducklake-catalog port=5432",
)

con = duckdb.connect()
for ext in ("parquet", "postgres", "ducklake"):
    try:
        con.execute(f"install {ext}")
        con.execute(f"load {ext}")
    except Exception as exc:  # noqa: BLE001 - surface, don't abort
        print(f"extension {ext}: {exc}")

# Attach the DuckLake catalog read-only in spirit: we only issue count(*).
lake_ok = False
try:
    con.execute(f"attach 'ducklake:{CATALOG}' as lake")
    lake_ok = True
except Exception as exc:  # noqa: BLE001
    print(f"Could not attach DuckLake catalog — silver/gold catalog tables skipped: {exc}")


def count_parquet(pattern: str) -> int | None:
    """Row count across parquet files matching a glob; None if no files / unreadable.
    union_by_name handles the football files' schema drift (7–106 columns)."""
    try:
        return con.execute(
            f"select count(*) from read_parquet('{pattern}', union_by_name=true)"
        ).fetchone()[0]
    except Exception:  # noqa: BLE001 - missing files / bad glob -> not present
        return None


rows: list[dict] = []

## Bronze — raw Parquet on disk
One row per top-level source under `data/bronze/` (each source's Parquet files are
counted recursively).

In [ ]:
bronze = DATA_DIR / "bronze"
if bronze.is_dir():
    for child in sorted(bronze.iterdir()):
        if child.is_dir():
            n = count_parquet(f"{child}/**/*.parquet")
        elif child.suffix == ".parquet":
            n = count_parquet(str(child))
        else:
            continue
        rows.append(
            {
                "layer": "bronze",
                "source": f"bronze/{child.name}",
                "location": "parquet",
                "records": n,
            }
        )
else:
    print(f"{bronze} does not exist")

pd.DataFrame([r for r in rows if r["layer"] == "bronze"])

## Silver & Gold — DuckLake catalog tables/views + on-disk Parquet
Every table and view in the `lake` catalog (schema name → layer:
`staging`/`intermediate` → silver, `marts` → gold), plus the Python-written
enrichment Parquet in `data/silver/` and the gold Parquet exports in `data/gold/`.

In [ ]:
if lake_ok:
    catalog_objects = con.execute(
        """
        select table_schema, table_name, table_type
        from information_schema.tables
        where table_catalog = 'lake'
        order by table_schema, table_name
        """
    ).fetchall()
    for schema, table, ttype in catalog_objects:
        s = schema.lower()
        # Fold both the current (staging/intermediate/marts) and any legacy
        # (silver/gold) dbt schema names into their medallion layer.
        if "staging" in s or "intermediate" in s or "silver" in s:
            layer = "silver"
        elif "mart" in s or "gold" in s:
            layer = "gold"
        else:
            layer = schema
        try:
            n = con.execute(f'select count(*) from lake."{schema}"."{table}"').fetchone()[0]
        except Exception:  # noqa: BLE001
            n = None
        rows.append(
            {
                "layer": layer,
                "source": f"{schema}.{table}",
                "location": f"ducklake ({ttype.lower()})",
                "records": n,
            }
        )

# Python-written silver enrichment Parquet + gold Parquet exports (on disk).
for layer_name in ("silver", "gold"):
    layer_dir = DATA_DIR / layer_name
    if layer_dir.is_dir():
        for f in sorted(layer_dir.glob("*.parquet")):
            rows.append(
                {
                    "layer": layer_name,
                    "source": f"{layer_name}/{f.name}",
                    "location": "parquet",
                    "records": count_parquet(str(f)),
                }
            )

pd.DataFrame([r for r in rows if r["layer"] not in ("bronze",)])

## Summary — full inventory + per-layer totals

In [ ]:
inventory = (
    pd.DataFrame(rows, columns=["layer", "source", "location", "records"])
    .sort_values(["layer", "records"], ascending=[True, False], na_position="last")
    .reset_index(drop=True)
)

totals = (
    inventory.groupby("layer", as_index=False)["records"]
    .sum()
    .rename(columns={"records": "total_records"})
)
print("Records per layer (double-counts across layers by design):")
print(totals.to_string(index=False))
print(f"\n{len(inventory)} sources total")

inventory